# Block 01: Claim Mapping and Execution Rules

**Goal**  
Freeze the experiment-to-claim map against the current paper checklist before numerical work.

**Checklist alignment**  
Maps E1-E12 to regimes, metrics, and artifact destinations.

**Outputs**  
- tables under `results/tables/`
- figures under `results/figures/`
- manifests under `results/manifests/`
- executed notebook copy under `results/notebooks/` when this suite is run with `--execute`

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "experiment_checklist.md").exists() and (candidate / "implementation").exists():
            return candidate
    raise RuntimeError("Could not locate repo root for notebook execution.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from implementation.notebook_support import *

cfg = CanonicalConfig().validate()
dirs = ensure_results_dirs(cfg)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

In [ ]:
claim_map = dataframe_from_claim_map()
naming_rules = pd.DataFrame(
    [
        ("preprocessing", "baseline_mean__rfft__one_sided_psd"),
        ("simulator_rank1", "QPSimulator aligned arrivals + external stationary noise"),
        ("simulator_family", "QPSimulator controlled tau/t0/n_QP family"),
        ("rank_label", "k{integer}"),
        ("seed_label", "seed_{integer}"),
        ("result_notebook", "results/notebooks/block_XX_*.ipynb"),
    ],
    columns=["name", "value"],
)
display(claim_map)
display(naming_rules)

In [ ]:
claim_map_path = dirs["tables"] / "block_01_claim_map.csv"
rules_path = dirs["manifests"] / "block_01_execution_rules.json"
save_dataframe(claim_map, claim_map_path)
save_json(
    {
        "canonical_config": config_as_frame(cfg).to_dict(orient="records"),
        "naming_rules": naming_rules.to_dict(orient="records"),
    },
    rules_path,
)
pd.DataFrame(
    [
        manifest_row("block_01", "governance", str(claim_map_path.relative_to(REPO_ROOT)), cfg),
        manifest_row("block_01", "governance", str(rules_path.relative_to(REPO_ROOT)), cfg),
    ]
)